In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

Path('../outputs').mkdir(parents=True, exist_ok=True)
N_BOOTSTRAP = 5_000

## Goal

Verify that the top 10 districts (largest private-school arithmetic advantage) and bottom 15 districts (government schools at parity or better) reported in the findings paper are not statistical artefacts of small sample sizes.

For each district:
1. Pull reported figures from `district_summary.csv`
2. Run a **Welch's t-test** on individual `arithmetic_score` values (government vs private) from `child_merged.csv`
3. Compute a **95% bootstrap confidence interval** on the gap (pvt − govt) with 5,000 resamples
4. **Flag** any district where the CI crosses zero — the direction of the gap cannot be asserted at the 5% level, and that district should not appear in a headline ranking

In [ ]:
TOP10 = [
    'Barkhan', 'Tando Muhammad Khan', 'Kurram Agency', 'Tank', 'Kohlu',
    'South Waziristan Agency', 'Sanghar', 'Dadu', 'Harnai', 'Orakzai Agency',
]
BOTTOM15 = [
    'Faisalabad', 'Sahiwal', 'Sheikhupura', 'Okara', 'Diamer', 'Lodhran',
    'Haripur', 'Muzaffargarh', 'Multan', 'Kachhi', 'Mirpur', 'Sialkot',
    'Chakwal', 'Dera Ghazi Khan', 'Jhang',
]
ALL_DISTRICTS = TOP10 + BOTTOM15

# Reported figures from district summary
ds = pd.read_csv('../Data:/district_summary.csv')
ds_sub = (
    ds[ds['district'].isin(ALL_DISTRICTS)]
    .set_index('district')
    [['province', 'n_government', 'n_private',
      'govt_mean_arithmetic', 'pvt_mean_arithmetic', 'arithmetic_gap']]
)

# Child-level scores — load only 3 columns from the 41 MB file
child = pd.read_csv(
    '../Data:/child_merged.csv',
    usecols=['district', 'school_type', 'arithmetic_score'],
)
child = child[
    child['district'].isin(ALL_DISTRICTS) &
    child['school_type'].isin(['Government', 'Private'])
].dropna(subset=['arithmetic_score'])

# Match diagnostics
not_in_summary = [d for d in ALL_DISTRICTS if d not in ds_sub.index]
not_in_child   = [d for d in ALL_DISTRICTS if d not in child['district'].unique()]
if not_in_summary:
    print('Districts missing from district_summary.csv:', not_in_summary)
if not_in_child:
    print('Districts missing from child_merged.csv:', not_in_child)
if not not_in_summary and not not_in_child:
    print(f'All {len(ALL_DISTRICTS)} districts matched in both files.')
print(f'Child records in target districts: {len(child):,}')

## Methodology

**Welch's t-test**: does not assume equal variances — appropriate here because government school samples are typically 3–10× larger than private school samples within a district.

**Bootstrap CI**: percentile method, 5,000 resamples. Within each resample, government and private samples are drawn independently with replacement at their original sizes, preserving group structure. The 2.5th and 97.5th percentiles of the 5,000 resampled gaps give the 95% CI.

**Flag criterion**: `ci_crosses_zero = True` if `ci_low < 0 < ci_high`. This means the data are consistent with the gap being zero, and the district should not appear in a directional headline ranking.

In [ ]:
def bootstrap_gap_ci(govt_scores, pvt_scores, n_boot=N_BOOTSTRAP, seed=42):
    """Percentile bootstrap 95% CI for (pvt_mean - govt_mean)."""
    rng   = np.random.default_rng(seed)
    g, p  = np.asarray(govt_scores), np.asarray(pvt_scores)
    idx_g = rng.integers(0, len(g), (n_boot, len(g)))
    idx_p = rng.integers(0, len(p), (n_boot, len(p)))
    gaps  = p[idx_p].mean(axis=1) - g[idx_g].mean(axis=1)
    return float(np.percentile(gaps, 2.5)), float(np.percentile(gaps, 97.5))


def check_district(district, list_label, ds_sub, child):
    """Return one result dict for district, or None if data are insufficient."""
    d_child = child[child['district'] == district]
    govt_sc = d_child[d_child['school_type'] == 'Government']['arithmetic_score'].values
    pvt_sc  = d_child[d_child['school_type'] == 'Private']['arithmetic_score'].values

    if len(govt_sc) < 5 or len(pvt_sc) < 5:
        print(f'  SKIP {district}: n_govt={len(govt_sc)}, n_pvt={len(pvt_sc)} — insufficient')
        return None

    t_stat, p_val = stats.ttest_ind(pvt_sc, govt_sc, equal_var=False)
    ci_low, ci_high = bootstrap_gap_ci(govt_sc, pvt_sc)
    ci_crosses_zero = ci_low < 0 < ci_high

    # Use reported district-summary figures where available
    if district in ds_sub.index:
        row      = ds_sub.loc[district]
        province = str(row['province'])
        n_govt   = int(row['n_government'])
        n_pvt    = int(row['n_private'])
        gap      = float(row['arithmetic_gap'])
        govt_m   = float(row['govt_mean_arithmetic'])
        pvt_m    = float(row['pvt_mean_arithmetic'])
    else:
        province = 'Unknown'
        n_govt   = len(govt_sc)
        n_pvt    = len(pvt_sc)
        govt_m   = float(govt_sc.mean())
        pvt_m    = float(pvt_sc.mean())
        gap      = pvt_m - govt_m

    flag = '  ← CI CROSSES ZERO ⚠' if ci_crosses_zero else ''
    print(f'  {district:<28} gap={gap:+.3f}  p={p_val:.4f}  '
          f'95%CI=[{ci_low:+.3f}, {ci_high:+.3f}]{flag}')

    return {
        'district':        district,
        'list':            list_label,
        'province':        province,
        'n_govt':          n_govt,
        'n_pvt':           n_pvt,
        'govt_mean':       round(govt_m, 3),
        'pvt_mean':        round(pvt_m, 3),
        'gap':             round(gap, 3),
        't_stat':          round(t_stat, 3),
        't_pvalue':        round(p_val, 4),
        'ci_low':          round(ci_low, 3),
        'ci_high':         round(ci_high, 3),
        'ci_crosses_zero': ci_crosses_zero,
    }

In [ ]:
results = []

print('── TOP 10  (largest private advantage) ──')
for d in TOP10:
    r = check_district(d, 'top10', ds_sub, child)
    if r:
        results.append(r)

print()
print('── BOTTOM 15  (government ≥ private) ──')
for d in BOTTOM15:
    r = check_district(d, 'bottom15', ds_sub, child)
    if r:
        results.append(r)

results_df = pd.DataFrame(results)

In [ ]:
out_path = '../outputs/district_outlier_check.csv'
results_df.to_csv(out_path, index=False)
print(f'Saved → {out_path}\n')

display_cols = [
    'district', 'province', 'n_govt', 'n_pvt',
    'gap', 't_pvalue', 'ci_low', 'ci_high', 'ci_crosses_zero',
]

top10_df  = results_df[results_df['list'] == 'top10'][display_cols]
bot15_df  = results_df[results_df['list'] == 'bottom15'][display_cols]

print('TOP 10 — private advantage districts')
print(top10_df.to_string(index=False))
print()
print('BOTTOM 15 — government advantage districts')
print(bot15_df.to_string(index=False))

In [ ]:
top10_res  = results_df[results_df['list'] == 'top10']
bot15_res  = results_df[results_df['list'] == 'bottom15']

top10_flagged = top10_res[top10_res['ci_crosses_zero']]
bot15_flagged = bot15_res[bot15_res['ci_crosses_zero']]

top10_ok = len(top10_res) - len(top10_flagged)
bot15_ok = len(bot15_res) - len(bot15_flagged)

# Build verdict
if len(top10_flagged) == 0:
    top10_clause = (
        f'all {len(top10_res)} survive the CI check — '
        f'every reported private advantage is statistically distinguishable from zero'
    )
else:
    names = ', '.join(top10_flagged['district'].tolist())
    top10_clause = (
        f'{top10_ok} of {len(top10_res)} survive; '
        f'{len(top10_flagged)} district(s) have a CI that crosses zero '
        f'({names}) and should be removed from the headline list'
    )

if len(bot15_flagged) == 0:
    bot15_clause = (
        f'all {len(bot15_res)} survive — '
        f'every reported government advantage (or parity) is statistically '
        f'distinguishable from a positive private advantage'
    )
else:
    names = ', '.join(bot15_flagged['district'].tolist())
    bot15_clause = (
        f'{bot15_ok} of {len(bot15_res)} survive; '
        f'{len(bot15_flagged)} district(s) have a CI that crosses zero '
        f'({names}) and should be treated as statistical parity, not a confirmed government advantage'
    )

verdict = (
    f'Outlier check verdict (5,000-resample bootstrap, 95% CI, Welch\'s t-test): '
    f'of the top 10 highest-gap districts, {top10_clause}. '
    f'Of the bottom 15 government-advantage districts, {bot15_clause}. '
    f'Results are saved to outputs/district_outlier_check.csv.'
)

print(verdict)